In [3]:
#==========================================================
# Load Packages 
#==========================================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from equiformer_pytorch import Equiformer
from torch_geometric.utils import to_dense_batch, to_dense_adj

In [4]:
#======================================================================
# Build Equiformer model
#======================================================================

#-------------------------------------------------------------
# Define Equiformer 
#-------------------------------------------------------------
class EquiformerQM9(nn.Module):
    def __init__(self, n_token=11, n_out=19, hidden_dim=128):
        super().__init__()

        self.hidden_dim = hidden_dim

        # 1) Atom feature embedding -> hidden_dim
        self.embedding = nn.Linear(n_token, hidden_dim)

        # 2) Equiformer core
        # input_degrees=1: inputs are scalar features
        # num_degrees=2: internal features include degree 0 and 1 (scalars + vectors)
        self.model = Equiformer(
            dim=hidden_dim,
            dim_in=hidden_dim,
            input_degrees=1,
            num_degrees=2,

            heads=4,
            dim_head=hidden_dim // 4,   # 32 when hidden_dim=128
            depth=1, # 8 stacked Equiformer blocks.Each block typically contains: equivariant graph attention, feed-forward / gating, equivariant normalization, residuals.

            # --- key efficiency / "molecular graph" knobs ---
            attend_sparse_neighbors=True,  # requires adj_mat
            num_neighbors=0,               # 0 = bonds only; >0 adds closest geometric neighbors
            num_adj_degrees_embed=2,       # adds 2-hop connectivity embedding
            max_sparse_neighbors=16,       # cap total sparse neighbors

            # we generally don't need valid_radius if we pass adj_mat,
            # but if num_neighbors > 0 it will still use geometry to fetch nearest neighbors.
            valid_radius=5.0,

            reduce_dim_out=False,
            attend_self=True,
            l2_dist_attention=False
        )

        # 3) Regression head
        self.linear = nn.Linear(hidden_dim, n_out) # After we get a molecule-level representation (a vector per molecule), we predict 19 targets.

    def forward(self, data):
        x, coords, batch = data.x, data.pos, data.batch

        # 1) Embed node features
        x = self.embedding(x)

        # 2) Dense batching : This is necessary because attention is usually implemented on dense tensors.
        x, mask = to_dense_batch(x, batch)          # (B, N, F), (B, N)
        coords, _ = to_dense_batch(coords, batch)   # (B, N, 3)

        # 3) Build adjacency from bond graph (PyG edge_index)
        #    to_dense_adj returns float 0/1, convert to bool
        """
        We pass the bond adjacency matrix to Equiformer to constrain attention to chemically meaningful neighbors.
        This ensures that information always propagates along covalent bonds,while geometric neighbors are added to capture non-bonded interactions.
        """
        adj_mat = to_dense_adj(data.edge_index, batch=batch).bool()  # (B, N, N)

        # 4) Forward through Equiformer using sparse neighbor attention
        out = self.model(x, coords, mask=mask, adj_mat=adj_mat)

        # 5) Extract invariant (degree-0) features safely
        if hasattr(out, "type0"):
            x = out.type0
        elif isinstance(out, dict):
            x = out.get(0, next(iter(out.values())))
        elif isinstance(out, (list, tuple)):
            x = out[0]
        else:
            x = out

        # 6) Pool if needed and predict
        if x.ndim == 2:
            # (B, F) already pooled
            return self.linear(x)

        if x.ndim == 3:
            # (B, N, F) node features -> masked mean pooling
            mask_f = mask[:, :x.size(1)].float()
            x = (x * mask_f.unsqueeze(-1)).sum(dim=1) / mask_f.sum(dim=1, keepdim=True).clamp_min(1.0)
            return self.linear(x)

        raise ValueError(f"Unexpected output shape from Equiformer: {x.shape}")